# STAGE 3: Accuracy-Focused Models
## Goal: Find highest-accuracy model (sacrifice some speed for better accuracy)

This notebook covers:
1. Training Gradient Boosting, XGBoost, and MLP
2. Comparing all 4 models (RF from Stage 2 + 3 new models)
3. Ranking by accuracy metrics
4. Identifying best accuracy model


## Section 1: Import Libraries and Load Stage 2 Data

In [9]:
import pandas as pd
import numpy as np
import pickle
import json
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import xgboost as xgb
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, classification_report
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load data from Stage 1
stage1_data = pickle.load(open('D:\sem_7\Research - Copy\stage_experiments\Stage_1_Data_Prep\stage1_preprocessed_data.pkl', 'rb'))
X_train_scaled = stage1_data['X_train_scaled']
X_test_scaled = stage1_data['X_test_scaled']
y_train = stage1_data['y_train']
y_test = stage1_data['y_test']
X_columns = stage1_data['X_columns']

# Load original unscaled data for XGBoost (it handles scaling internally)
stage1_data_unscaled = pickle.load(open('D:\sem_7\Research - Copy\stage_experiments\Stage_1_Data_Prep\stage1_preprocessed_data.pkl', 'rb'))
X_train_unscaled = stage1_data_unscaled['X_train_scaled']  # Note: already using scaled if that's what was saved
X_test_unscaled = stage1_data_unscaled['X_test_scaled']

# Load Random Forest from Stage 2
rf = pickle.load(open('D:\sem_7\Research - Copy\stage_experiments\Stage_2_Fast_Models\stage2_random_forest_model.pkl', 'rb'))
stage2_preds = pickle.load(open('D:\sem_7\Research - Copy\stage_experiments\Stage_2_Fast_Models\stage2_predictions.pkl', 'rb'))
y_prob_rf = stage2_preds['y_prob_rf']

print(f"✓ Data loaded, {X_train_scaled.shape[0]} training samples, {X_test_scaled.shape[0]} test samples")

✓ Data loaded, 1820732 training samples, 780314 test samples


## Section 2: Model 1 - Gradient Boosting

**What is Gradient Boosting?**
- Builds trees sequentially
- Each new tree corrects errors of previous trees
- Focuses learning on hard-to-classify samples
- Better accuracy than Random Forest (94-96% vs 91-92%)

In [ ]:
print("Training Gradient Boosting...")
gb = GradientBoostingClassifier(
    n_estimators=200,          # Number of boosting stages
    learning_rate=0.1,         # Shrinkage rate
    max_depth=5,               # Shallow trees prevent overfitting
    random_state=42,
    verbose=0
)
gb.fit(X_train_scaled, y_train)
y_prob_gb = gb.predict_proba(X_test_scaled)[:, 1]
y_pred_gb = gb.predict(X_test_scaled)

metrics_gb = {
    'Model': 'Gradient Boosting',
    'F1-Score': f1_score(y_test, y_pred_gb),
    'Precision': precision_score(y_test, y_pred_gb),
    'Recall': recall_score(y_test, y_pred_gb),
    'ROC-AUC': roc_auc_score(y_test, y_prob_gb)
}
print(f"  F1-Score: {metrics_gb['F1-Score']:.4f}")

Training Gradient Boosting...


## Section 3: Model 2 - XGBoost

**What is XGBoost (eXtreme Gradient Boosting)?**
- Optimized version of Gradient Boosting
- 20-50% faster training than GB
- Better regularization prevents overfitting
- Typically 1-2% better accuracy than GB (95-97% F1)

In [ ]:
print("Training XGBoost...")

# Calculate class weights for imbalance
scale_pos_weight = sum(y_train == 0) / sum(y_train == 1)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=5,
    scale_pos_weight=scale_pos_weight,  # Weight for class imbalance
    random_state=42,
    verbosity=0
)
xgb_model.fit(X_train_unscaled, y_train)  # XGBoost handles scaling
y_prob_xgb = xgb_model.predict_proba(X_test_unscaled)[:, 1]
y_pred_xgb = xgb_model.predict(X_test_unscaled)

metrics_xgb = {
    'Model': 'XGBoost',
    'F1-Score': f1_score(y_test, y_pred_xgb),
    'Precision': precision_score(y_test, y_pred_xgb),
    'Recall': recall_score(y_test, y_pred_xgb),
    'ROC-AUC': roc_auc_score(y_test, y_prob_xgb)
}
print(f"  F1-Score: {metrics_xgb['F1-Score']:.4f}")

Training XGBoost...
  F1-Score: 0.9999


## Section 4: Model 3 - MLP (Neural Network)

**What is MLP (Multi-Layer Perceptron)?**
- Artificial neural network with hidden layers
- Learns complex non-linear patterns
- Feature interactions learned automatically
- Typically 92-94% F1 for WBAN

In [ ]:
print("Training MLP...")
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),  # 3 hidden layers
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=1000,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1
)
mlp.fit(X_train_scaled, y_train)
y_prob_mlp = mlp.predict_proba(X_test_scaled)[:, 1]
y_pred_mlp = mlp.predict(X_test_scaled)

metrics_mlp = {
    'Model': 'MLP',
    'F1-Score': f1_score(y_test, y_pred_mlp),
    'Precision': precision_score(y_test, y_pred_mlp),
    'Recall': recall_score(y_test, y_pred_mlp),
    'ROC-AUC': roc_auc_score(y_test, y_prob_mlp)
}
print(f"  F1-Score: {metrics_mlp['F1-Score']:.4f}")

Training MLP...
  F1-Score: 0.9997


## Section 5: Model 4 - Random Forest (Reference from Stage 2)

In [ ]:
# Get Random Forest metrics
y_pred_rf = rf.predict(X_test_scaled)

metrics_rf = {
    'Model': 'Random Forest',
    'F1-Score': f1_score(y_test, y_pred_rf),
    'Precision': precision_score(y_test, y_pred_rf),
    'Recall': recall_score(y_test, y_pred_rf),
    'ROC-AUC': roc_auc_score(y_test, y_prob_rf)
}
print(f"Random Forest F1-Score: {metrics_rf['F1-Score']:.4f}")

Random Forest F1-Score: 0.9999


## Section 6: Compare All Models

Create comprehensive comparison of all 4 models.

In [ ]:
# Create comparison dataframe
all_metrics = pd.DataFrame([metrics_rf, metrics_gb, metrics_xgb, metrics_mlp])
all_metrics_sorted = all_metrics.sort_values('F1-Score', ascending=False)

print("="*80)
print("STAGE 3: ACCURACY-FOCUSED MODELS COMPARISON")
print("="*80)
print(all_metrics_sorted.to_string(index=False))
print("="*80)

# Save results
all_metrics_sorted.to_csv('stage3_accuracy_comparison.csv', index=False)
print("\n Comparison saved to stage3_accuracy_comparison.csv")

# Identify best model
best_model_name = all_metrics_sorted.iloc[0]['Model']
best_f1 = all_metrics_sorted.iloc[0]['F1-Score']
print(f"\n BEST MODEL: {best_model_name} with F1-Score: {best_f1:.4f}")

STAGE 3: ACCURACY-FOCUSED MODELS COMPARISON
            Model  F1-Score  Precision   Recall  ROC-AUC
Gradient Boosting  0.999917   0.999966 0.999869 1.000000
          XGBoost  0.999867   0.999908 0.999826 1.000000
    Random Forest  0.999864   1.000000 0.999728 1.000000
              MLP  0.999673   0.999936 0.999409 0.999992

 Comparison saved to stage3_accuracy_comparison.csv

 BEST MODEL: Gradient Boosting with F1-Score: 0.9999


## Section 7: Go/No-Go Decision for Stage 4

Decide whether to proceed with ensemble and optimization.

In [ ]:
print("="*80)
print("STAGE 3: GO/NO-GO DECISION")
print("="*80)

if best_f1 > 0.94:
    print(f"\n DECISION: EXCELLENT - PROCEED TO STAGE 4 (Ensemble)")
    print(f"  Best model: {best_model_name}")
    print(f"  F1-Score: {best_f1:.4f} (>94%)")
    print(f"  Next: Build ensemble and optimize hyperparameters")
elif best_f1 > 0.90:
    print(f"\n DECISION: GOOD - PROCEED TO STAGE 4")
    print(f"  Best model: {best_model_name}")
    print(f"  F1-Score: {best_f1:.4f}")
    print(f"  Ensemble may provide marginal improvement")
else:
    print(f"\n DECISION: MARGINAL - CONSIDER FEATURE ENGINEERING")
    print(f"  Best F1-Score: {best_f1:.4f}")
    print(f"  May need to improve features before ensemble")

STAGE 3: GO/NO-GO DECISION

 DECISION: EXCELLENT - PROCEED TO STAGE 4 (Ensemble)
  Best model: Gradient Boosting
  F1-Score: 0.9999 (>94%)
  Next: Build ensemble and optimize hyperparameters


## Section 8: Save Models and Results

In [ ]:
# Save models
pickle.dump(gb, open('stage3_gradient_boosting_model.pkl', 'wb'))
pickle.dump(xgb_model, open('stage3_xgboost_model.pkl', 'wb'))
pickle.dump(mlp, open('stage3_mlp_model.pkl', 'wb'))

# Save predictions for Stage 4
stage3_predictions = {
    'y_prob_rf': y_prob_rf,
    'y_prob_gb': y_prob_gb,
    'y_prob_xgb': y_prob_xgb,
    'y_prob_mlp': y_prob_mlp
}
pickle.dump(stage3_predictions, open('stage3_predictions.pkl', 'wb'))

# Save metrics
metrics_dict = {
    'best_model': best_model_name,
    'best_f1': float(best_f1),
    'all_metrics': all_metrics_sorted.to_dict()
}
json.dump(metrics_dict, open('stage3_results.json', 'w'), indent=2)

print(" Results saved:")
print(f"  - stage3_gradient_boosting_model.pkl")
print(f"  - stage3_xgboost_model.pkl")
print(f"  - stage3_mlp_model.pkl")
print(f"  - stage3_predictions.pkl")
print(f"  - stage3_accuracy_comparison.csv")
print(f"  - stage3_results.json")

 Results saved:
  - stage3_gradient_boosting_model.pkl
  - stage3_xgboost_model.pkl
  - stage3_mlp_model.pkl
  - stage3_predictions.pkl
  - stage3_accuracy_comparison.csv
  - stage3_results.json


## Summary

1. Trained 3 accuracy-focused models
2. Compared all 4 models systematically
3. Identified best accuracy model

**Next Step:** Proceed to **Stage 4: Ensemble & Optimization**
- Hyperparameter tune best model
- Build voting ensemble
- Expected: 96-98% F1-score
""".format(
        best_model_name,
        best_f1 * 100,
        metrics_gb['F1-Score'] * 100,
        metrics_xgb['F1-Score'] * 100,
        metrics_mlp['F1-Score'] * 100,
        metrics_rf['F1-Score'] * 100
    )